# 3.1.3: corridas de evaluación, bloque 3 de 4 (MO-07, MO-08)

Las 30 corridas son 10 meta-órdenes × 3 semillas, repartidas en 4 cuadernos. Cada uno corre su bloque con el **sistema final** (Maestro y Ejecutor PPO, de `sistema_final`) y con **TWAP y VWAP** en las mismas órdenes, semillas e impacto acumulado. Córrelos **después** de `Colab_sistema_final`, de a dos al mismo tiempo.

**Cómo usarlo:** «Ejecutar todas». Dura ≈ 2 h 15 min. Si se corta, vuelve a ejecutar todo: retoma donde quedó.

Queda en **`corridas_30/bloque_3`**. La última celda arma la tabla con todos los bloques que ya estén listos.

## Preparación del entorno (celdas 1 a 9)

In [ ]:
# CELDA 1 (reinicia el entorno solo; el aviso "sesión fallada" es normal, no la repitas)
!pip install -q "condacolab<0.2"
import condacolab
condacolab.install()

In [ ]:
# CELDA 2
import os
if not os.path.exists('/usr/local/envs/abides'):
    !conda create -n abides python=3.9 -y -q
!conda run -n abides python --version

In [ ]:
# CELDA 3 (unos minutos; un aviso de Pillow es normal)
%env PIP_DISABLE_PIP_VERSION_CHECK=1
!conda run -n abides python -m pip install -q pip==23.3.2 setuptools==57.5.0 wheel==0.38.4
!conda run -n abides python -m pip install -q "cython<3" numpy==1.22.0
!conda run -n abides python -m pip install -q --no-build-isolation --no-deps gym==0.18.0
!conda run -n abides python -m pip install -q --no-build-isolation numpy==1.22.0 "cloudpickle>=1.2,<1.7" "pyglet>=1.4,<=1.5.0" "Pillow<10" pandas==1.2.4 scipy==1.10.0 coloredlogs==15.0.1 psutil==5.8.0 tqdm==4.61.1 pomegranate==0.14.5 "ray[tune]==1.7.0" "protobuf==3.20.3" pyarrow==11.0.0 matplotlib==3.5.3 pytest
!conda run -n abides python -m pip install -q "gymnasium==0.29.1" "importlib-metadata>=4.8" farama-notifications --no-deps

In [ ]:
# CELDA 4 (deben salir tres "OK")
import os
if not os.path.exists('/content/abides-jpmc-public'):
    !git clone -q https://github.com/jpmorganchase/abides-jpmc-public.git /content/abides-jpmc-public
for pkg in ['abides-core', 'abides-markets', 'abides-gym']:
    !cd /content/abides-jpmc-public/{pkg} && conda run -n abides python setup.py -q install > /content/install_{pkg}.log 2>&1 && echo "{pkg} OK" || (echo "{pkg} FALLO"; tail -20 /content/install_{pkg}.log)

In [ ]:
# CELDA 5 (debe imprimir 0.18.0 1.22.0 1.2.4 0.29.1 1.7.0)
!conda run -n abides python -c "import gym, numpy, pandas, gymnasium, ray, abides_core, abides_markets, abides_gym; print(gym.__version__, numpy.__version__, pandas.__version__, gymnasium.__version__, ray.__version__)"

In [ ]:
# CELDA 6 (torch; debe imprimir torch 1.13.1+cpu | numpy 1.22.0)
!conda run -n abides python -m pip install -q torch==1.13.1+cpu --extra-index-url https://download.pytorch.org/whl/cpu
!conda run -n abides python -c "import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__)"

In [ ]:
# CELDA 7 (código del proyecto: rama feature/ps-cierre-sprint6; debe mostrar un commit y un número mayor que 0)
import os
if not os.path.exists('/content/slippage'):
    !git clone -q https://github.com/Vatelfil/slippage.git /content/slippage
!cd /content/slippage && git fetch -q origin && git checkout -q -f origin/feature/ps-cierre-sprint6
!cd /content/slippage && git log --oneline -1 && grep -c "imitar_heuristica" scripts/colab/entrenamiento_inicial_ps.py


In [ ]:
# CELDA 8 (conecta Drive; acepta el permiso)
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/slippage_colab/resultados'
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
print(RESULTS_DIR)

In [ ]:
# CELDA 9 (seaborn, que necesita el módulo de benchmarks)
!conda run -n abides python -m pip install -q seaborn==0.12.2 --no-deps

## Sistema final

In [ ]:
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/evaluar_sistema_ps.py --meta-ordenes data/meta_ordenes/meta_ordenes_v2_candidata.json --entorno abides --ordenes MO-07 MO-08 --seeds 3 --impacto --out-dir {RESULTS_DIR}/corridas_30/bloque_3 --max-minutes 330 --modo muestreo --ckpt-maestro {RESULTS_DIR}/sistema_final/maestro/checkpoint_last.pt --ckpt-ejecutor apertura={RESULTS_DIR}/sistema_final/maestro/ejecutor_apertura.pt --ckpt-ejecutor media_jornada={RESULTS_DIR}/sistema_final/maestro/ejecutor_media_jornada.pt --ckpt-ejecutor cierre={RESULTS_DIR}/sistema_final/maestro/ejecutor_cierre.pt --nombre PPO_sistema 2>&1 | grep -v UserWarning


## TWAP y VWAP

In [ ]:
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/benchmarks_meta_ordenes_ps.py --meta-ordenes data/meta_ordenes/meta_ordenes_v2_candidata.json --entorno abides --ordenes MO-07 MO-08 --seeds 3 --impacto --out-dir {RESULTS_DIR}/corridas_30/bloque_3 --max-minutes 330 2>&1 | grep -v UserWarning


## Tabla con los bloques listos (media, desviación, IC 95 %, Wilcoxon y t-test)

In [ ]:
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/consolidar_resultados_3_2_2.py --entradas {RESULTS_DIR}/corridas_30/bloque_1 {RESULTS_DIR}/corridas_30/bloque_2 {RESULTS_DIR}/corridas_30/bloque_3 {RESULTS_DIR}/corridas_30/bloque_4 --sistema PPO_sistema --out-dir {RESULTS_DIR}/corridas_30/tabla
print(open(f'{RESULTS_DIR}/corridas_30/tabla/tabla_maestra.md', encoding='utf-8').read())


## Zip para descargar

In [ ]:
import os, zipfile
base = f'{RESULTS_DIR}/corridas_30'
archivos = [os.path.join(c, a) for c, _, fs in os.walk(base) for a in fs if a.endswith(('.json', '.csv', '.md', '.png'))]
zpath = f'{RESULTS_DIR}/corridas_30_bloque_3.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for a in archivos:
        z.write(a, os.path.relpath(a, RESULTS_DIR))
print(len(archivos), 'archivos -> zip listo:', zpath)
